In [1]:
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.runnables import RunnablePassthrough



In [2]:
#Splitting documents into chunk
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = splitter.create_documents([
    "Machine learning is a subset of artificial intelligence.",
    "RAG combines retrieval with LLM generation."
])

#Creating embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

#storing vector in ChromaDB
vectorstore = Chroma.from_documents(
    docs,
    embeddings
)

# Retrieve top 2 relevant documents from vector database
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

import os

os.environ["GROQ_API_KEY"] = "your API key"

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)


C:\Users\Admin\AppData\Local\Temp\ipykernel_14892\3594947795.py:12: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:

# Prompt template combining retrieved context, conversation history, and user question

prompt = ChatPromptTemplate.from_template("""
You are a helpful AI assistant.

Use the following context to answer the question.

Context:
{context}

Previous Conversation:
{history}

Question:
{question}
""")

# Store last 3 conversations for contextual memory
memory = ConversationBufferWindowMemory(
    k=3,  
    return_messages=True
)


# Build RAG pipeline using retrieved context, memory history, prompt template, and LLM
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough(),
        "history": lambda x: memory.load_memory_variables({})["history"]
    }
    | prompt
    | llm
)



C:\Users\Admin\AppData\Local\Temp\ipykernel_14892\2330565873.py:19: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferWindowMemory(


In [4]:

# Continuous conversation loop with memory-aware response generation
while True:
    user_input = input("you:")
    if user_input.lower()== 'exit':
        break
    response = rag_chain.invoke(user_input)
    print("bot:",response.content)
    memory.save_context(
    {"input": user_input},
    {"output": response.content}
)

you: How do I proceed to the next stage?


bot: To proceed to the next stage, I would need more information about the current stage you are referring to. The context provided seems to be related to artificial intelligence and machine learning, but it doesn't give me enough details to provide a specific answer.

Could you please provide more context or clarify what you mean by "the next stage"? Are you referring to a specific process, project, or task? I'll do my best to assist you once I have a better understanding of your question.


you: exit
